# Лабораторная работа №18: Быстрые samplers и distillation компактной diffusion model

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Переиспользовать checkpoint lab 17.
- Сравнить DDPM, DDIM и дополнительный solver.
- Обучить compact student и построить Pareto frontier.


## Теоретическая часть

DDIM использует тот же denoiser, но может идти по сокращённой последовательности timesteps детерминированно. Distillation обучает student воспроизводить несколько teacher steps одним шагом. Сравнение строится при одинаковых initial noise и labels.


### Математическая постановка

Deterministic DDIM update использует оценку $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\epsilon_\theta)/\sqrt{\bar\alpha_t}$. Distillation: $\mathcal L=\|S_\phi(x_t,t)-T^{(k)}_\theta(x_t,t)\|^2$.


### Материалы

- [DDIM](https://arxiv.org/abs/2010.02502)
- [Progressive Distillation](https://arxiv.org/abs/2202.00512)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Переиспользовать checkpoint lab 17.
2. Сравнить DDPM, DDIM и дополнительный solver.
3. Обучить compact student и построить Pareto frontier.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** teacher lab 17 + DDIM; distillation на MNIST/2D.
- **Ограничение данных:** фиксированный eval batch.
- **Fallback:** DDIM без distillation на CPU.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


## Реализация


In [ ]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = True  # False — полный эксперимент
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


In [ ]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Данные и подготовка

**Постановка.** Разделение подготовки данных от моделирования — базовое требование к воспроизводимым экспериментам: детерминированная загрузка, фиксированные преобразования и контролируемая генерация батчей позволяют атрибутировать изменение метрик изменениям модели, а не случайным различиям пайплайна.

Выполните следующие действия:

1. Реализуйте необходимые функции: `make_schedule(total, steps)`.
2. Выполните последовательность вычислительных шагов (одна итерация); все случайные величины фиксируйте через ранее объявленный `SEED`.
3. Сохраните результат в `artifacts/./lab_17_ddpm_cfg/artifacts/ddpm.pt` и продублируйте ключевые числа в ячейке вывода отдельной инструкцией вывода.
4. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода и `artifacts/./lab_17_ddpm_cfg/artifacts/ddpm.pt`.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/../lab_17_ddpm_cfg/artifacts/ddpm.pt
# Артефакт: Сохраните сигнатуры и формат артефактов, описанные выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Основной пайплайн

**Постановка.** Реализация, полученная в настоящем задании, образует вычислительное ядро практикума: от её корректности зависит валидность всех последующих измерений.

Выполните следующие действия:

1. Реализуйте необходимые функции: `ddim_step(x, t, t_prev)`, `sample_ddim(model, noise, labels)`.
2. Выполните последовательность вычислительных шагов (одна итерация); все случайные величины фиксируйте через ранее объявленный `SEED`.
3. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
4. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните сигнатуры и формат артефактов, описанные выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит обучаемых параметров

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Добавьте quality metric (classifier accuracy/FID-like feature distance) и diversity.
2. Student distillation: регрессия выхода student к результату нескольких teacher updates.
3. Постройте Pareto frontier: минимальное время при не меньшем качестве.

**Результат.** Результаты задания в ячейке вывода.

**Критерий принятия.** Задание считается принятым, если все пункты плана исполнены в этом ноутбуке: ячейка отрабатывает без ошибок, вывод виден в ячейке вывода, требуемое сохранено.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Основной пайплайн

*Вес: 0.75 балла.*

**Постановка.** Реализация, полученная в настоящем задании, образует вычислительное ядро практикума: от её корректности зависит валидность всех последующих измерений.

Выполните следующие действия:

1. Выполните последовательность вычислительных шагов (одна итерация); все случайные величины фиксируйте через ранее объявленный `SEED`.
2. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
3. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Для steps=[5,10,25,50,100] проверьте: результат отсортирован по убыванию, длины <= steps, все значения < T. Артефакт: artifacts/schedule_invariants.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит обучаемых параметров

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Выполните требуемые вычисления одноразовым сценарием; все случайные величины фиксируйте через ранее объявленный `SEED`.
2. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
3. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Проверьте: при eps=0 и t=t_prev x не меняется; при больших |eps| выход отличается; форма сохраняется. Сохраните artifacts/ddim_invariants.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит обучаемых параметров

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Реализуйте необходимые функции: `counted18(x18, t18, y18)`, `guided18(x18, t18, y18)`.
2. Выполните требуемые вычисления одноразовым сценарием; все случайные величины фиксируйте через ранее объявленный `SEED`.
3. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
4. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Для steps=[5,25,50] вычислите количество обращений к guided_fn (подсчёт через счётчик-обёртку); оно должно равняться steps. Сохраните artifacts/callcount.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит обучаемых параметров

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Выполните требуемые вычисления одноразовым сценарием; все случайные величины фиксируйте через ранее объявленный `SEED`.
2. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
3. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Дважды выполните sample_ddim с одинаковым noise: результат должен совпадать. Сохраните artifacts/ddim_det.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Основной пайплайн

*Вес: 0.75 балла.*

**Постановка.** Реализация, полученная в настоящем задании, образует вычислительное ядро практикума: от её корректности зависит валидность всех последующих измерений.

Выполните следующие действия:

1. Соберите: доступность checkpoint'а, инварианты, вызовы guided_fn, детерминизм — в artifacts/sampler_summary.json (assert на ключи).

**Критерий принятия.** Задание считается принятым, если соберите: доступность checkpoint'а, инварианты, вызовы guided_fn, детерминизм — в artifacts/sampler_summary.json (assert на ключи).
.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

In [ ]:
# Самопроверка: все обязательные ячейки выше должны быть исполнены (кроме бонусной), в outputs не должно быть ошибок.
# Если какая-то ячейка не исполнена или упала - вернитесь и доведите её до конца.
check(True, "самопроверка: обязательные ячейки исполнены, ошибок нет")

### Инженерия результатов

Ниже — усложнённые задания обязательной части. Они не повторяют базовые, а достраивают вокруг результата лабы инженерный контур: стресс-тест, статистическая значимость, воспроизводимость, стоимость и сводный дашборд. Выполняются по порядку: последнее собирает итог из предыдущих. Вес каждого — 1 балл.

### Стресс-тест: качество при возмущении входа

**Вес: 1 балл, обязательное.** Возьмите основной результат лабы как «счётчик качества» и постройте кривую его деградации при нарастающих возмущениях (несколько уровней, повторы на каждый). Ключевой вопрос — порог: при какой силе возмущения результат перестаёт быть пригодным. Артефакт: `artifacts/stress_robustness.json`.

In [ ]:
# Контекст лабы: lab 18 fast samplers distillation. Счётчик качества — число из результатов этой лабы (метрика, IoU, accuracy, качественная оценка).
# Что нужно сделать:
# 1) определить количественный счётчик качества по результатам лабы;
# 2) построить его зависимость от силы возмущения входа (3-5 уровней, по 3 повтора на уровень);
# 3) записать кривую (сила, среднее, std) и относительную деградацию;
# 4) вывод: при какой силе возмущения результат ещё приемлем.
# Артефакт: artifacts/stress_robustness.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Доверительные интервалы: бутстреп и значимость разницы

**Вес: 1 балл, обязательное.** Одного среднего недостаточно: превратите результат в выборку испытаний, постройте 95% бутстреп-интервал для основной конфигурации и для ослабленной, проверьте, пересекаются ли интервалы. Отдельно посмотрите, как ширина интервала зависит от числа ресэплов. Артефакт: `artifacts/bootstrap_ci.json`.

In [ ]:
# Контекст лабы: lab 18 fast samplers distillation.
# Что нужно сделать:
# 1) превратить основной результат в выборку испытаний (успех/неудача) или числовых значений;
# 2) бутстреп (>=400 ресэплов) 95% CI для основной и для ослабленной конфигурации (например, x0.9);
# 3) пересекаются ли интервалы - вывод о значимости разницы;
# 4) бонус-наблюдение: как ширина CI зависит от числа ресэплов (100/400/1600).
# Артефакт: artifacts/bootstrap_ci.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Воспроизводимость: разброс по случайным зернам (seed)

**Вес: 1 балл, обязательное.** Повторите ключевой эксперимент лабы при нескольких состояниях генератора случайных чисел (если полный запуск дорогой — на подвыборке), оцените стандартное отклонение и ответьте: можно ли доверять одиночному запуску. Артефакт: `artifacts/seed_variance.json`.

In [ ]:
# Контекст лабы: lab 18 fast samplers distillation.
# Что нужно сделать:
# 1) повторить ключевой эксперимент лабы при 4 разных seed (если дорого - на подвыборке);
# 2) записать пары (seed, score) и посчитать стандартное отклонение;
# 3) вывод: можно ли доверять одиночному запуску; что конкретно улучшит воспроизводимость (фиксация генераторов, детерминизм,фиксация генераторов...);
# 4) отдельно: какие операции в лабе недетерминированы.
# Артефакт: artifacts/seed_variance.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Стоимость: время, память и вес артефактов

**Вес: 1 балл, обязательное.** Замерьте, во что обходится результат: время ключевого прогона, пик памяти (tracemalloc), суммарный вес артефактов. Найдите главный вкладчик и постройте оценку, как стоимость растёт при десятикратном увеличении данных. Артефакт: `artifacts/cost_profile.json`.

In [ ]:
# Контекст лабы: lab 18 fast samplers distillation.
# Что нужно сделать:
# 1) замерить время ключевого прогона, пик памяти (tracemalloc) и суммарный вес артефактов;
# 2) найти самого тяжёлого вкладчика (по времени/памяти/размеру);
# 3) прогноз: как стоимость растёт при увеличении данных в 10 раз (по какому компоненту - линейно, квадратично);
# 4) вывод: главное вычислительное узкое место лабы.
# Артефакт: artifacts/cost_profile.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Сводный дашборд: все проверки лабы в одном отчёте

**Вес: 1 балл, обязательное.** Соберите результаты всех предыдущих проверок и артефакты лабы в один отчёт: словарь флагов PASS/WARN/MISSING по каждому направлению, машиночитаемый JSON и человекочитаемый markdown. Это итоговая самодиагностика лабораторной. Артефакты: `artifacts/final_dashboard.json`, `artifacts/final_dashboard.md`.

In [ ]:
# Контекст лабы: lab 18 fast samplers distillation. Задание собирает итог из предыдущих - выполняется последним.
# Что нужно сделать:
# 1) прочитать json-результаты всех проверок выше (и остальные артефакты лабы);
# 2) выставить по каждому направлению флаг PASS/WARN/MISSING с порогами, которые вы объявите явно;
# 3) сохранить artifacts/final_dashboard.json (машиночитаемый) и artifacts/final_dashboard.md (человекочитаемый);
# 4) резюме лабы в трёх строках: что работает, что осторожно, что не проверено.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит обучаемых параметров

*Вес: 0.5 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Соберите список всех файлов в каталоге artifacts/, для каждого — размер в байтах; убедитесь, что ни один не пуст.

**Результат.** Сохраните artifacts/files_audit.json.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните artifacts/files_audit.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Данные и подготовка

*Вес: 0.5 балла.*

**Постановка.** Разделение подготовки данных от моделирования — базовое требование к воспроизводимым экспериментам: детерминированная загрузка, фиксированные преобразования и контролируемая генерация батчей позволяют атрибутировать изменение метрик изменениям модели, а не случайным различиям пайплайна.

Выполните следующие действия:

1. Соберите версии ключевых библиотек (torch, numpy и др. через importlib.metadata) и общий путь к данным DATA_ROOT если определён.

**Результат.** Сохраните artifacts/env_inventory.json; выполняйте assert, что словарь непуст.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните artifacts/env_inventory.json; выполняйте assert, что словарь непуст.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит обучаемых параметров

*Вес: 0.5 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Гипотеза.
2. Эксперимент с фиксированным seed.
3. Метрика и сравнение.
4. Вывод: подтвердилась/нет, почему.

**Критерий принятия.** Задание считается принятым, если вывод: подтвердилась/нет, почему, результат виден в ячейке вывода в этом ноутбуке и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

## Анализ результатов

Заполните после запуска:

1. Таблица сравниваемых конфигураций и вычислительный бюджет.
2. Основные метрики с доверительными интервалами или вариативностью по seed, где это возможно.
3. Не менее пяти характерных ошибок/неудачных примеров.
4. Ablation: какой компонент действительно дал эффект?
5. Ограничения эксперимента и следующий проверяемый шаг.


In [ ]:
# Автоматическая проверка наличия каталога артефактов
assert ARTIFACTS.exists()
print("Artifacts:", [p.name for p in ARTIFACTS.iterdir()])


## Выводы

Сформулируйте 3–5 выводов, каждый из которых опирается на измерение или наблюдаемый пример. Не подменяйте вывод перечислением выполненных действий.
